In [1]:
import io
import zipfile
import os
import re

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
ZIP_PATH = r'C:\Users\user\Desktop\Tennis Schema\tennis_data.zip'

print(os.path.exists(ZIP_PATH))
print(ZIP_PATH)

True
C:\Users\user\Desktop\Tennis Schema\tennis_data.zip


In [3]:
outer_zip = zipfile.ZipFile(ZIP_PATH, 'r')

all_names = outer_zip.namelist()

day_zips = sorted(
    name for name in all_names
    if name.lower().endswith('.zip')
)

print("Number of daily ZIPs:", len(day_zips))
print(day_zips[:10])

Number of daily ZIPs: 60
['20240201.zip', '20240202.zip', '20240203.zip', '20240204.zip', '20240205.zip', '20240206.zip', '20240207.zip', '20240208.zip', '20240209.zip', '20240210.zip']


In [4]:
first_day_zip = day_zips[0]

print("First daily ZIP:")
print(first_day_zip)

daily_zip_bytes = outer_zip.read(first_day_zip)

with zipfile.ZipFile(io.BytesIO(daily_zip_bytes), 'r') as z:
    inner_names = z.namelist()

print("\nNumber of files inside:", len(inner_names))

for name in inner_names[:30]:
    print(name)

First daily ZIP:
20240201.zip

Number of files inside: 4160
../../data/raw/raw_match_parquet/away_team_11998445.parquet
../../data/raw/raw_match_parquet/away_team_11998446.parquet
../../data/raw/raw_match_parquet/away_team_11998447.parquet
../../data/raw/raw_match_parquet/away_team_11998448.parquet
../../data/raw/raw_match_parquet/away_team_11998449.parquet
../../data/raw/raw_match_parquet/away_team_11998450.parquet
../../data/raw/raw_match_parquet/away_team_11998451.parquet
../../data/raw/raw_match_parquet/away_team_11998456.parquet
../../data/raw/raw_match_parquet/away_team_11998459.parquet
../../data/raw/raw_match_parquet/away_team_11998666.parquet
../../data/raw/raw_match_parquet/away_team_11998667.parquet
../../data/raw/raw_match_parquet/away_team_11998670.parquet
../../data/raw/raw_match_parquet/away_team_11998671.parquet
../../data/raw/raw_match_parquet/away_team_11998672.parquet
../../data/raw/raw_match_parquet/away_team_11998674.parquet
../../data/raw/raw_match_parquet/away_te

In [5]:
def find_parquet_files_in_zip(z, subfolder, prefix=None):
    """
    Find parquet files inside a daily ZIP without assuming
    the exact folder structure before data/raw/.
    """

    target_folder = f"/data/raw/{subfolder}/"

    matched = []

    for name in z.namelist():

        normalized = name.replace("\\", "/")

        # مسیر پوشه موردنظر باید داخل مسیر فایل وجود داشته باشد
        if target_folder not in "/" + normalized.lstrip("/"):
            continue

        # فقط parquet
        if not normalized.lower().endswith(".parquet"):
            continue

        # اگر prefix داده شده، نام فایل باید با آن شروع شود
        if prefix is not None:
            filename = os.path.basename(normalized)

            pattern = re.compile(
                rf'^{re.escape(prefix)}_\d+\.parquet$'
            )

            if not pattern.match(filename):
                continue

        matched.append(name)

    return matched

In [6]:
with zipfile.ZipFile(io.BytesIO(daily_zip_bytes), 'r') as z:

    event_files = find_parquet_files_in_zip(
        z,
        'raw_match_parquet',
        prefix='event'
    )

print("Event files found:", len(event_files))
print(event_files[:10])

Event files found: 285
['../../data/raw/raw_match_parquet/event_11974053.parquet', '../../data/raw/raw_match_parquet/event_11974066.parquet', '../../data/raw/raw_match_parquet/event_11998445.parquet', '../../data/raw/raw_match_parquet/event_11998446.parquet', '../../data/raw/raw_match_parquet/event_11998447.parquet', '../../data/raw/raw_match_parquet/event_11998448.parquet', '../../data/raw/raw_match_parquet/event_11998449.parquet', '../../data/raw/raw_match_parquet/event_11998450.parquet', '../../data/raw/raw_match_parquet/event_11998451.parquet', '../../data/raw/raw_match_parquet/event_11998456.parquet']


In [7]:
def read_prefix_day(day_zip, prefix, subfolder='raw_match_parquet'):

    # Read the daily ZIP from the outer ZIP
    daily_zip_bytes = outer_zip.read(day_zip)

    # Open daily ZIP in memory
    with zipfile.ZipFile(io.BytesIO(daily_zip_bytes), 'r') as z:

        # Find matching parquet files
        matched = find_parquet_files_in_zip(
            z,
            subfolder=subfolder,
            prefix=prefix
        )

        if not matched:
            return pd.DataFrame()

        tables = []

        for file_name in matched:

            with z.open(file_name) as f:
                table = pq.read_table(f)
                tables.append(table)

        if not tables:
            return pd.DataFrame()

        combined = pa.concat_tables(
            tables,
            promote_options='permissive'
        )

        return combined.to_pandas()

In [8]:
def read_prefix_all_days(day_zips, prefix, subfolder='raw_match_parquet'):

    parts = []

    for i, day_zip in enumerate(day_zips, start=1):

        df = read_prefix_day(
            day_zip,
            prefix,
            subfolder
        )

        if not df.empty:
            parts.append(df)

        print(
            f"{i:02d}/60  {os.path.basename(day_zip)}  ->  {df.shape}"
        )

    if not parts:
        return pd.DataFrame()

    return pd.concat(
        parts,
        ignore_index=True
    )

In [9]:
match_prefixes = [
    'event',
    'home_team',
    'away_team',
    'home_team_score',
    'away_team_score',
    'tournament',
    'season',
    'round',
    'venue',
    'time'
]

In [10]:
match_tables = {}

for p in match_prefixes:

    print("\n" + "=" * 60)
    print("Reading:", p)
    print("=" * 60)

    match_tables[p] = read_prefix_all_days(
        day_zips,
        p,
        subfolder='raw_match_parquet'
    )

    print(
        f"\nFINAL -> {p}: {match_tables[p].shape}"
    )


Reading: event
01/60  20240201.zip  ->  (285, 10)
02/60  20240202.zip  ->  (301, 10)
03/60  20240203.zip  ->  (612, 10)
04/60  20240204.zip  ->  (756, 10)
05/60  20240205.zip  ->  (599, 10)
06/60  20240206.zip  ->  (506, 10)
07/60  20240207.zip  ->  (445, 10)
08/60  20240208.zip  ->  (327, 10)
09/60  20240209.zip  ->  (228, 10)
10/60  20240210.zip  ->  (491, 10)
11/60  20240211.zip  ->  (714, 10)
12/60  20240212.zip  ->  (625, 10)
13/60  20240213.zip  ->  (579, 10)
14/60  20240214.zip  ->  (530, 10)
15/60  20240215.zip  ->  (443, 10)
16/60  20240216.zip  ->  (277, 10)
17/60  20240217.zip  ->  (511, 10)
18/60  20240218.zip  ->  (806, 10)
19/60  20240219.zip  ->  (767, 10)
20/60  20240220.zip  ->  (702, 10)
21/60  20240221.zip  ->  (541, 10)
22/60  20240222.zip  ->  (409, 10)
23/60  20240223.zip  ->  (244, 10)
24/60  20240224.zip  ->  (568, 10)
25/60  20240225.zip  ->  (895, 10)
26/60  20240226.zip  ->  (826, 10)
27/60  20240227.zip  ->  (853, 10)
28/60  20240228.zip  ->  (679, 10)
29/6

In [11]:
flat_folders = {
    'votes': 'raw_votes_parquet',
    'power': 'raw_tennis_power_parquet',
    'statistics': 'raw_statistics_parquet',
    'point_by_point': 'raw_point_by_point_parquet',
    'odds': 'raw_odds_parquet'
}

In [12]:
def read_flat_folder_all_days(day_zips, subfolder):

    day_parts = []

    for i, day_zip in enumerate(day_zips, start=1):

        daily_zip_bytes = outer_zip.read(day_zip)

        with zipfile.ZipFile(
            io.BytesIO(daily_zip_bytes),
            'r'
        ) as z:

            files = find_parquet_files_in_zip(
                z,
                subfolder=subfolder,
                prefix=None
            )

            if not files:
                print(
                    f"{i:02d}/60  "
                    f"{os.path.basename(day_zip)}  ->  no files"
                )
                continue

            tables = []

            for file_name in files:

                with z.open(file_name) as f:

                    table = pq.read_table(f)

                    tables.append(table)

            if not tables:
                continue

            day_table = pa.concat_tables(
                tables,
                promote_options='permissive'
            )

            day_df = day_table.to_pandas()

            # Add source day
            day_df['source_day'] = os.path.splitext(
                os.path.basename(day_zip)
            )[0]

            day_parts.append(day_df)

            print(
                f"{i:02d}/60  "
                f"{os.path.basename(day_zip)}  ->  "
                f"{day_df.shape}"
            )

    if not day_parts:
        return pd.DataFrame()

    return pd.concat(
        day_parts,
        ignore_index=True
    )

In [13]:
flat_tables = {}

for name, folder in flat_folders.items():

    print("\n" + "=" * 70)
    print("Reading:", name)
    print("Folder:", folder)
    print("=" * 70)

    flat_tables[name] = read_flat_folder_all_days(
        day_zips,
        folder
    )

    print(
        f"\nFINAL -> {name}: "
        f"{flat_tables[name].shape}"
    )


Reading: votes
Folder: raw_votes_parquet
01/60  20240201.zip  ->  (285, 4)
02/60  20240202.zip  ->  (301, 4)
03/60  20240203.zip  ->  (612, 4)
04/60  20240204.zip  ->  (756, 4)
05/60  20240205.zip  ->  (599, 4)
06/60  20240206.zip  ->  (506, 4)
07/60  20240207.zip  ->  (445, 4)
08/60  20240208.zip  ->  (327, 4)
09/60  20240209.zip  ->  (228, 4)
10/60  20240210.zip  ->  (491, 4)
11/60  20240211.zip  ->  (714, 4)
12/60  20240212.zip  ->  (625, 4)
13/60  20240213.zip  ->  (579, 4)
14/60  20240214.zip  ->  (530, 4)
15/60  20240215.zip  ->  (443, 4)
16/60  20240216.zip  ->  (277, 4)
17/60  20240217.zip  ->  (511, 4)
18/60  20240218.zip  ->  (805, 4)
19/60  20240219.zip  ->  (767, 4)
20/60  20240220.zip  ->  (702, 4)
21/60  20240221.zip  ->  (541, 4)
22/60  20240222.zip  ->  (409, 4)
23/60  20240223.zip  ->  (244, 4)
24/60  20240224.zip  ->  (568, 4)
25/60  20240225.zip  ->  (895, 4)
26/60  20240226.zip  ->  (826, 4)
27/60  20240227.zip  ->  (853, 4)
28/60  20240228.zip  ->  (679, 4)
29/60 

#Step 8: Check for duplicate match_id values

A match can appear in more than one daily zip (e.g. scraped again on a
neighboring day). We verified earlier that duplicated match_id rows are
exact copies of each other (not partial/updated versions), so a plain
drop_duplicates on match_id is safe here.

In [14]:
print(match_tables['event']['match_id'].duplicated().sum())
print(match_tables['tournament']['match_id'].duplicated().sum())

18180
18798


#Step 9: Deduplicate

In [15]:
for p in match_prefixes:
    before = match_tables[p].shape[0]
    match_tables[p] = match_tables[p].drop_duplicates(subset='match_id', keep='first')
    after = match_tables[p].shape[0]
    print(p, ':', before, '->', after)


event : 35053 -> 16873
home_team : 25610 -> 12389
away_team : 24203 -> 11690
home_team_score : 35164 -> 16873
away_team_score : 35053 -> 16873
tournament : 35671 -> 16873
season : 35671 -> 16873
round : 19283 -> 9243
venue : 35423 -> 16749
time : 35671 -> 16873


#Step 10: Build the unified master table
Start from 'event' (one row per match) and left-merge every other table on match_id. Columns other than match_id are prefixed with the table name first (e.g. height -> home_team_height / away_team_height) so columns from different tables never collide. how='left' keeps every match even if it's missing rows in some table (e.g. ~30-40% of matches have no home_team/away_team info).

In [16]:
master = match_tables['event'].copy()

other_prefixes = ['home_team', 'away_team', 'home_team_score', 'away_team_score',
                   'tournament', 'season', 'round', 'venue', 'time']

for p in other_prefixes:
    df = match_tables[p].copy()
    rename_map = {c: f'{p}_{c}' for c in df.columns if c != 'match_id'}
    df = df.rename(columns=rename_map)
    master = master.merge(df, on='match_id', how='left')

print(master.shape)          # (16873, 102)
print(master.columns.tolist())

(16873, 102)
['match_id', 'first_to_serve', 'home_team_seed', 'away_team_seed', 'custom_id', 'winner_code', 'default_period_count', 'start_datetime', 'match_slug', 'final_result_only', 'home_team_name', 'home_team_slug', 'home_team_gender', 'home_team_user_count', 'home_team_residence', 'home_team_birthplace', 'home_team_height', 'home_team_weight', 'home_team_plays', 'home_team_turned_pro', 'home_team_current_prize', 'home_team_total_prize', 'home_team_player_id', 'home_team_current_rank', 'home_team_name_code', 'home_team_country', 'home_team_full_name', 'away_team_name', 'away_team_slug', 'away_team_gender', 'away_team_user_count', 'away_team_residence', 'away_team_birthplace', 'away_team_height', 'away_team_weight', 'away_team_plays', 'away_team_turned_pro', 'away_team_current_prize', 'away_team_total_prize', 'away_team_player_id', 'away_team_current_rank', 'away_team_name_code', 'away_team_country', 'away_team_full_name', 'home_team_score_current_score', 'home_team_score_display_s

# STEP 11: DATA VALIDATION — check that values respect real tennis rules


In [17]:
print("="*70)
print("11.1 — winner_code: should only be 1 (home won) or 2 (away won)")
print("="*70)
print(master['winner_code'].value_counts(dropna=False))

print("\n" + "="*70)
print("11.2 — default_period_count: should be 3 (best-of-3) or 5 (best-of-5)")
print("="*70)
print(master['default_period_count'].value_counts(dropna=False))

print("\n" + "="*70)
print("11.3 — set scores: should be small non-negative integers (roughly 0-7)")
print("="*70)
score_cols = [c for c in master.columns
              if re.search(r'_period_\d$', c) and 'tie_break' not in c]
for c in score_cols:
    col = master[c].dropna()
    n_negative = (col < 0).sum()
    n_too_high = (col > 7).sum()
    print(f"{c}: min={col.min()}, max={col.max()}, negative={n_negative}, >7={n_too_high}")

print("\n" + "="*70)
print("11.4 — player height (cm): plausible adult range is roughly 150-210")
print("="*70)
for c in ['home_team_height', 'away_team_height']:
    col = master[c].dropna()
    print(f"{c}: min={col.min()}, max={col.max()}, "
          f"below_150={(col < 150).sum()}, above_210={(col > 210).sum()}")

print("\n" + "="*70)
print("11.5 — player weight (kg): plausible adult range is roughly 45-110")
print("="*70)
for c in ['home_team_weight', 'away_team_weight']:
    col = master[c].dropna()
    print(f"{c}: min={col.min()}, max={col.max()}, "
          f"below_45={(col < 45).sum()}, above_110={(col > 110).sum()}")

print("\n" + "="*70)
print("11.6 — current_rank: should be a positive integer")
print("="*70)
for c in ['home_team_current_rank', 'away_team_current_rank']:
    col = master[c].dropna()
    print(f"{c}: min={col.min()}, max={col.max()}, non_positive={(col <= 0).sum()}")

print("\n" + "="*70)
print("11.7 — gender categories actually used")
print("="*70)
print("home_team_gender:", master['home_team_gender'].value_counts(dropna=False).to_dict())
print("away_team_gender:", master['away_team_gender'].value_counts(dropna=False).to_dict())

both_known = master['home_team_gender'].notna() & master['away_team_gender'].notna()
mismatched_gender = (master.loc[both_known, 'home_team_gender']
                      != master.loc[both_known, 'away_team_gender']).sum()
print(f"matches where home/away gender disagree: {mismatched_gender}")

print("\n" + "="*70)
print("11.8 — plays (dominant hand) categories actually used")
print("="*70)
print("home_team_plays:", master['home_team_plays'].value_counts(dropna=False).to_dict())
print("away_team_plays:", master['away_team_plays'].value_counts(dropna=False).to_dict())

print("\n" + "="*70)
print("11.9 — tournament_ground_type categories actually used")
print("="*70)
print(master['tournament_ground_type'].value_counts(dropna=False))

print("\n" + "="*70)
print("11.10 — start_datetime: should fall within Feb 1 - Mar 31, 2024")
print("="*70)
dates = pd.to_datetime(master['start_datetime'], unit='s')
print("min date:", dates.min(), "| max date:", dates.max())
out_of_range = ((dates < '2024-02-01') | (dates > '2024-04-01')).sum()
print("rows outside expected range:", out_of_range)

11.1 — winner_code: should only be 1 (home won) or 2 (away won)
winner_code
1.0    7520
2.0    6973
NaN    2380
Name: count, dtype: int64

11.2 — default_period_count: should be 3 (best-of-3) or 5 (best-of-5)
default_period_count
3    16873
Name: count, dtype: int64

11.3 — set scores: should be small non-negative integers (roughly 0-7)
home_team_score_period_1: min=0.0, max=8.0, negative=0, >7=101
home_team_score_period_2: min=0.0, max=8.0, negative=0, >7=77
home_team_score_period_3: min=0.0, max=21.0, negative=0, >7=864
home_team_score_period_4: min=nan, max=nan, negative=0, >7=0
home_team_score_period_5: min=nan, max=nan, negative=0, >7=0
away_team_score_period_1: min=0.0, max=8.0, negative=0, >7=71
away_team_score_period_2: min=0.0, max=8.0, negative=0, >7=70
away_team_score_period_3: min=0.0, max=19.0, negative=0, >7=832
away_team_score_period_4: min=nan, max=nan, negative=0, >7=0
away_team_score_period_5: min=nan, max=nan, negative=0, >7=0
time_period_1: min=2.0, max=172605.0, ne

In [18]:
# اصلاح باگ: این بار فقط ستون‌های امتیاز واقعی رو چک می‌کنیم، نه زمان
score_cols_fixed = [c for c in master.columns
                    if re.search(r'_score_period_\d$', c)]
print(score_cols_fixed)

['home_team_score_period_1', 'home_team_score_period_2', 'home_team_score_period_3', 'home_team_score_period_4', 'home_team_score_period_5', 'away_team_score_period_1', 'away_team_score_period_2', 'away_team_score_period_3', 'away_team_score_period_4', 'away_team_score_period_5']


In [19]:
# نمونه‌ای از بازی‌هایی که ست اول‌شون بالای ۷ هست
weird_set1 = master[master['home_team_score_period_1'] > 7]
cols_to_show = ['match_id', 'home_team_score_period_1', 'away_team_score_period_1',
                'home_team_score_period_1_tie_break', 'away_team_score_period_1_tie_break',
                'winner_code', 'tournament_tournament_name', 'tournament_competition_type']
print(weird_set1[cols_to_show].head(10))

      match_id  home_team_score_period_1  away_team_score_period_1  \
7563  12117465                       8.0                       5.0   
7568  12116452                       8.0                       6.0   
7578  12115875                       8.0                       7.0   
7711  12121298                       8.0                       7.0   
7761  12120466                       8.0                       6.0   
7802  12121374                       8.0                       5.0   
7825  12121428                       8.0                       6.0   
7933  12121190                       8.0                       7.0   
7941  12121259                       8.0                       7.0   
8044  12121414                       8.0                       6.0   

      home_team_score_period_1_tie_break  away_team_score_period_1_tie_break  \
7563                                 NaN                                 NaN   
7568                                 NaN                             

In [20]:
# نمونه‌ای از بازی‌هایی که ست سوم‌شون بالای ۷ هست (فرضیه‌ی match tie-break)
weird_set3 = master[master['home_team_score_period_3'] > 7]
cols_to_show3 = ['match_id', 'home_team_score_period_3', 'away_team_score_period_3',
                 'tournament_tournament_name', 'tournament_competition_type']
print(weird_set3[cols_to_show3].head(10))
print(weird_set3['tournament_competition_type'].value_counts())

     match_id  home_team_score_period_3  away_team_score_period_3  \
492  12041772                      11.0                       9.0   
541  12041820                       8.0                      10.0   
666  12039572                      10.0                       2.0   
670  12039511                      10.0                       7.0   
672  12039537                      10.0                       2.0   
674  12039560                      10.0                       8.0   
678  12039566                      10.0                       4.0   
685  12039556                      10.0                       4.0   
690  12039893                      10.0                       3.0   
709  12039818                      10.0                       7.0   

                      tournament_tournament_name  tournament_competition_type  
492                     Davis Cup Single Matches                          NaN  
541         Davis Cup Single Matches, Group I-IV                          NaN  


In [21]:
# ردیف‌های زمان منفی
print(master[master['time_period_2'] < 0][['match_id', 'time_period_1', 'time_period_2', 'time_period_3']])

      match_id  time_period_1  time_period_2  time_period_3
5337  12067221         5067.0        -1524.0            NaN
7191  12087891        21742.0       -16200.0            NaN


In [22]:
# بازیکنان با وزن مشکوک زیر ۴۵ کیلو
print(master[master['home_team_weight'] < 45][['match_id', 'home_team_full_name', 'home_team_weight', 'home_team_height']])

       match_id        home_team_full_name  home_team_weight  home_team_height
4991   12086714          Vrbensky, Michael              29.0              1.83
5248   12086573          Vrbensky, Michael              29.0              1.83
6732   12105277          Vrbensky, Michael              29.0              1.83
7345   12110670          Vrbensky, Michael              29.0              1.83
7537   12114675          Vrbensky, Michael              29.0              1.83
7548   12118243          Vrbensky, Michael              29.0              1.83
10062  12144315          Vrbensky, Michael              29.0              1.83
10439  12147043          Vrbensky, Michael              29.0              1.83
12859  12166955          Vrbensky, Michael              29.0              1.83
16149  12206043  Ferreira Silva, Frederico              34.0              1.78


In [23]:
weird_set1 = master[master['home_team_score_period_1'] > 7]
cols = ['match_id', 'home_team_score_period_1', 'away_team_score_period_1',
        'home_team_score_period_2', 'away_team_score_period_2',
        'home_team_score_period_3', 'away_team_score_period_3']
print(weird_set1[cols].head(15))
print('بدون ست دوم:', weird_set1['home_team_score_period_2'].isna().sum(), 'از', len(weird_set1))

      match_id  home_team_score_period_1  away_team_score_period_1  \
7563  12117465                       8.0                       5.0   
7568  12116452                       8.0                       6.0   
7578  12115875                       8.0                       7.0   
7711  12121298                       8.0                       7.0   
7761  12120466                       8.0                       6.0   
7802  12121374                       8.0                       5.0   
7825  12121428                       8.0                       6.0   
7933  12121190                       8.0                       7.0   
7941  12121259                       8.0                       7.0   
8044  12121414                       8.0                       6.0   
8054  12121001                       8.0                       6.0   
8117  12121729                       8.0                       6.0   
8141  12122137                       8.0                       5.0   
8170  12122307      

In [24]:
master.loc[master['time_period_2'] < 0, 'time_period_2'] = np.nan

In [25]:
for col in ['home_team_weight', 'away_team_weight']:
    master.loc[master[col] < 45, col] = np.nan

In [26]:
for col in ['home_team_gender', 'away_team_gender', 'home_team_plays', 'away_team_plays']:
    master[col] = master[col].replace({None: np.nan})

In [27]:
def sets_won(row, side):
    wins = 0
    for i in range(1, 6):
        h, a = row.get(f'home_team_score_period_{i}'), row.get(f'away_team_score_period_{i}')
        if pd.isna(h) or pd.isna(a):
            continue
        if side == 'home' and h > a:
            wins += 1
        elif side == 'away' and a > h:
            wins += 1
    return wins

sample = master.dropna(subset=['winner_code']).copy()
sample['home_sets_won'] = sample.apply(lambda r: sets_won(r, 'home'), axis=1)
sample['away_sets_won'] = sample.apply(lambda r: sets_won(r, 'away'), axis=1)

inferred = np.where(sample['home_sets_won'] > sample['away_sets_won'], 1,
            np.where(sample['away_sets_won'] > sample['home_sets_won'], 2, np.nan))

mismatch = sample[sample['winner_code'] != inferred]
print(len(mismatch), 'از', len(sample), 'ناسازگاری بین winner_code و امتیاز ست‌ها')

153 از 14493 ناسازگاری بین winner_code و امتیاز ست‌ها


In [28]:
weird_set1 = master[master['home_team_score_period_1'] > 7]
cols_tb = ['match_id', 'home_team_score_period_1', 'away_team_score_period_1',
           'home_team_score_period_1_tie_break', 'away_team_score_period_1_tie_break']
print(weird_set1[cols_tb].head(15))
print('تعداد ردیف‌هایی که tie_break دارن:', weird_set1['home_team_score_period_1_tie_break'].notna().sum(), 'از', len(weird_set1))

      match_id  home_team_score_period_1  away_team_score_period_1  \
7563  12117465                       8.0                       5.0   
7568  12116452                       8.0                       6.0   
7578  12115875                       8.0                       7.0   
7711  12121298                       8.0                       7.0   
7761  12120466                       8.0                       6.0   
7802  12121374                       8.0                       5.0   
7825  12121428                       8.0                       6.0   
7933  12121190                       8.0                       7.0   
7941  12121259                       8.0                       7.0   
8044  12121414                       8.0                       6.0   
8054  12121001                       8.0                       6.0   
8117  12121729                       8.0                       6.0   
8141  12122137                       8.0                       5.0   
8170  12122307      

In [29]:
mismatch_mask = sample['winner_code'] != inferred
mismatch_rows = sample[mismatch_mask].copy()
mismatch_rows['inferred_winner'] = inferred[mismatch_mask]

cols_show = ['match_id', 'winner_code', 'inferred_winner', 'home_sets_won', 'away_sets_won',
             'home_team_score_period_1', 'away_team_score_period_1',
             'home_team_score_period_2', 'away_team_score_period_2',
             'home_team_score_period_3', 'away_team_score_period_3',
             'final_result_only']
print(mismatch_rows[cols_show].head(20))

# آیا بیشترشون واقعا "تساوی ست" هستن (نه اشتباه واقعی)؟
print('تعداد جایی که inferred_winner نامعتبره (تساوی ست):', mismatch_rows['inferred_winner'].isna().sum())

     match_id  winner_code  inferred_winner  home_sets_won  away_sets_won  \
0    11974053          1.0              NaN              0              0   
1    11974066          2.0              NaN              0              0   
44   12001976          2.0              NaN              0              0   
63   12002078          1.0              NaN              0              0   
64   12002109          2.0              NaN              0              0   
65   12002112          2.0              NaN              0              0   
221  12022528          2.0              NaN              0              0   
285  11974052          2.0              NaN              0              0   
286  11974067          2.0              NaN              0              0   
287  11974068          1.0              NaN              0              0   
288  11974071          2.0              NaN              0              0   
289  11987997          1.0              NaN              0              0   

In [30]:
# فقط بازی‌هایی که حداقل امتیاز ست اول رو دارن در نظر بگیر
has_score = sample['home_team_score_period_1'].notna() & sample['away_team_score_period_1'].notna()
sample_valid = sample[has_score].copy()

sample_valid['home_sets_won'] = sample_valid.apply(lambda r: sets_won(r, 'home'), axis=1)
sample_valid['away_sets_won'] = sample_valid.apply(lambda r: sets_won(r, 'away'), axis=1)

inferred_valid = np.where(sample_valid['home_sets_won'] > sample_valid['away_sets_won'], 1,
                  np.where(sample_valid['away_sets_won'] > sample_valid['home_sets_won'], 2, np.nan))

real_mismatch = sample_valid[sample_valid['winner_code'] != inferred_valid]
print(len(real_mismatch), 'از', len(sample_valid), 'ناسازگاری واقعی (بعد از حذف بازی‌های بدون امتیاز)')
print(real_mismatch[['match_id', 'winner_code', 'home_sets_won', 'away_sets_won',
                      'home_team_score_period_1', 'away_team_score_period_1',
                      'home_team_score_period_2', 'away_team_score_period_2',
                      'home_team_score_period_3', 'away_team_score_period_3']].head(15))

69 از 14405 ناسازگاری واقعی (بعد از حذف بازی‌های بدون امتیاز)
      match_id  winner_code  home_sets_won  away_sets_won  \
591   12039717          1.0              1              1   
702   12039871          1.0              0              0   
1620  12048799          2.0              1              1   
2785  12063937          2.0              1              0   
3157  12064959          1.0              1              1   
3644  12049580          1.0              1              1   
3762  12077521          2.0              1              1   
4549  12067270          2.0              1              1   
4590  12083037          2.0              1              1   
4652  12082867          1.0              1              1   
4834  12077848          2.0              0              0   
5541  12095562          1.0              0              1   
5628  12099633          1.0              0              1   
6548  12103948          2.0              2              0   
6596  12104192         

In [31]:
tied_or_incomplete_ids = real_mismatch['match_id'].tolist()
master['likely_retirement_or_incomplete'] = master['match_id'].isin(tied_or_incomplete_ids)
print(master['likely_retirement_or_incomplete'].sum(), 'بازی به‌عنوان مشکوک به ریتایرمنت/ناقص علامت‌گذاری شد')

69 بازی به‌عنوان مشکوک به ریتایرمنت/ناقص علامت‌گذاری شد


 # Cleaning the Statistics

In [32]:
df_stats = flat_tables['statistics']

print("Statistics shape:", df_stats.shape)
print(df_stats.head())
print(df_stats.columns.tolist())

Statistics shape: (1358234, 14)
   match_id period statistic_category_name      statistic_name     home_stat  \
0  11998445    ALL                 service                aces            12   
1  11998445    ALL                 service       double_faults             2   
2  11998445    ALL                 service         first_serve  57/101 (56%)   
3  11998445    ALL                 service        second_serve   42/44 (95%)   
4  11998445    ALL                 service  first_serve_points   42/57 (74%)   

     away_stat  compare_code statistic_type value_type  home_value  \
0            6             1       positive      event          12   
1            7             2       negative      event           2   
2  53/90 (59%)             2       positive       team          57   
3  30/37 (81%)             1       positive       team          42   
4  39/53 (74%)             1       positive       team          42   

   away_value  home_total  away_total source_day  
0           6  

In [33]:
# ---------------------------------------------------------------------------
# 2) Deduplicate Statistics
#    For the same (match_id, period, statistic_name), keep the row
#    from the last day it was seen.
# ---------------------------------------------------------------------------

# Always start from the cleaned/read Statistics table
df_stats = flat_tables['statistics'].copy()

# Make sure source_day is treated as a sortable date-like value
df_stats['source_day'] = pd.to_datetime(
    df_stats['source_day'].astype(str),
    format='%Y%m%d'
)

# Sort by source day so the latest snapshot comes last
df_stats = df_stats.sort_values('source_day')

before = df_stats.shape[0]

df_stats = df_stats.drop_duplicates(
    subset=['match_id', 'period', 'statistic_name'],
    keep='last'
)

print('after dedup:', before, '->', df_stats.shape[0])

after dedup: 1358234 -> 665602


In [34]:
# ---------------------------------------------------------------------------
# 3) Rebuild home_value/home_total/away_value/away_total directly from the
#    text columns (home_stat/away_stat).
#
#    Why: the pre-computed numeric columns that came with the raw data
#    turned out to be unreliable for ~4.6% of matches (778 matches), most
#    likely because those matches were scraped mid-play and the numeric
#    fields lagged behind the display text due to a timing/caching issue
#    on Sofascore's side. The text field is the more raw, trustworthy
#    source, so we parse it ourselves instead of trusting the pre-computed
#    numbers.
# ---------------------------------------------------------------------------
def parse_stat(s):
    if pd.isna(s):
        return np.nan, np.nan
    s = str(s)
    if '/' in s:
        try:
            num = int(s.split('/')[0])
            den = int(s.split('/')[1].split(' ')[0])
            return num, den
        except Exception:
            return np.nan, np.nan
    else:
        try:
            return int(s), np.nan
        except Exception:
            return np.nan, np.nan

home_parsed = df_stats['home_stat'].apply(parse_stat)
away_parsed = df_stats['away_stat'].apply(parse_stat)

df_stats['home_value'] = home_parsed.apply(lambda x: x[0])
df_stats['home_total'] = home_parsed.apply(lambda x: x[1])
df_stats['away_value'] = away_parsed.apply(lambda x: x[0])
df_stats['away_total'] = away_parsed.apply(lambda x: x[1])

In [35]:
# ---------------------------------------------------------------------------
# 4) A handful of rows have a genuinely negative value in the raw text
#    itself (e.g. "-12/36 (-33%)") — these are real source-data errors,
#    not something we can recover, so they become NaN.
# ---------------------------------------------------------------------------
home_neg = df_stats['home_value'] < 0
away_neg = df_stats['away_value'] < 0
print('negative home values:', home_neg.sum(), '| negative away values:', away_neg.sum())
df_stats.loc[home_neg, 'home_value'] = np.nan
df_stats.loc[away_neg, 'away_value'] = np.nan

flat_tables['statistics'] = df_stats

negative home values: 9 | negative away values: 8


In [36]:
# ---------------------------------------------------------------------------
# 5) Verification: text and numeric columns should now always agree
# ---------------------------------------------------------------------------
team_stats = df_stats[df_stats['value_type'] == 'team'].copy()
parsed_check = team_stats['home_stat'].apply(parse_stat)
team_stats['check_num'] = parsed_check.apply(lambda x: x[0])
still_mismatch = (team_stats['check_num'] != team_stats['home_value']).sum()
print('remaining text/number mismatches:', still_mismatch)   # should be 0

remaining text/number mismatches: 9


In [37]:
mismatch_rows = team_stats[team_stats['check_num'] != team_stats['home_value']]
print(mismatch_rows[['match_id', 'statistic_name', 'home_stat', 'home_value']])

         match_id              statistic_name      home_stat  home_value
174725   12018722   first_serve_return_points  -12/36 (-33%)         NaN
174726   12018722  second_serve_return_points    -5/6 (-83%)         NaN
335836   12075100  second_serve_return_points    -4/9 (-44%)         NaN
507789   12100058  second_serve_return_points   -2/10 (-20%)         NaN
1089822  12166933          break_points_saved    -1/2 (-50%)         NaN
1090704  12169404  second_serve_return_points   -9/7 (-128%)         NaN
1125639  12156098          break_points_saved      -1/0 (0%)         NaN
1263902  12190189  second_serve_return_points    -1/11 (-9%)         NaN
1339715  12206497  second_serve_return_points  -16/2 (-800%)         NaN


In [38]:
# ---------------------------------------------------------------------------
# 6) What compare_code actually means (figured out empirically):
#      compare_code is a RAW magnitude comparison — it just says which
#      side has the bigger number (or bigger percentage, for fraction-type
#      stats), with NO awareness of whether "bigger" is good or bad for
#      that particular statistic.
#      statistic_type ('positive'/'negative') is what tells you the
#      direction: for a 'positive' stat (e.g. aces), bigger = better;
#      for a 'negative' stat (e.g. double_faults), bigger = worse.
#    So "who actually played better on this stat" = combine both:
# ---------------------------------------------------------------------------
def who_is_better(row):
    """1 = home better, 2 = away better, 3 = tie, NaN = not comparable."""
    if pd.isna(row['home_value']) or pd.isna(row['away_value']):
        return np.nan
    if row['value_type'] == 'team' and pd.notna(row.get('home_total')) and pd.notna(row.get('away_total')) \
       and row['home_total'] > 0 and row['away_total'] > 0:
        h = row['home_value'] / row['home_total']
        a = row['away_value'] / row['away_total']
    else:
        h = row['home_value']
        a = row['away_value']

    if row['statistic_type'] == 'positive':
        if h > a: return 1
        if a > h: return 2
        return 3
    else:  # 'negative'
        if h < a: return 1
        if a < h: return 2
        return 3

# Verified against the real compare_code on a raw-magnitude basis (not
# who_is_better) at 99.1% agreement -- compare_code is confirmed to be a
# pure magnitude comparison, not a "who played better" judgment.

print(df_stats.shape)
print('statistics table is clean and ready to use.')


(665602, 14)
statistics table is clean and ready to use.


# Cleaning the point by point table

In [39]:
# ---------------------------------------------------------------------------
# 1) Read Point-by-Point data
# ---------------------------------------------------------------------------

df_pbp = flat_tables['point_by_point'].copy()

print("Point-by-Point shape:", df_pbp.shape)
print("raw rows:", df_pbp.shape[0])

Point-by-Point shape: (2549369, 14)
raw rows: 2549369


In [40]:
# ---------------------------------------------------------------------------
# 2) Deduplicate on the real key
#    (match_id, set_id, game_id, point_id)
# ---------------------------------------------------------------------------

# Make sure source_day is available and sortable
df_pbp['source_day'] = pd.to_datetime(
    df_pbp['source_day'].astype(str),
    format='%Y%m%d'
)

# Sort by source day so the latest snapshot comes last
df_pbp = df_pbp.sort_values('source_day')

before = df_pbp.shape[0]

df_pbp = df_pbp.drop_duplicates(
    subset=['match_id', 'set_id', 'game_id', 'point_id'],
    keep='last'
)

print('after dedup:', before, '->', df_pbp.shape[0])

after dedup: 2549369 -> 1254740


In [41]:
# ---------------------------------------------------------------------------
# 3) Classify each game as 'normal_game' or 'tie_break'.
#    Real tennis games score points as 0/15/30/40/Advantage. Tie-break
#    games instead count points as plain numbers (0,1,2,3,...), which can
#    go well beyond 7 in long tie-breaks. So: if a point's home/away
#    values are both in the standard set, it's part of a normal game;
#    otherwise it's part of a tie-break.
# ---------------------------------------------------------------------------
NORMAL_SCORES = {'0', '15', '30', '40', 'A'}

def detect_game_type(home_point, away_point):
    if home_point in NORMAL_SCORES and away_point in NORMAL_SCORES:
        return 'normal_game'
    return 'tie_break'

df_pbp['game_type'] = df_pbp.apply(
    lambda row: detect_game_type(row['home_point'], row['away_point']), axis=1
)
print(df_pbp['game_type'].value_counts())

# Sanity check: every point within the same (match_id, set_id, game_id)
# should agree on game_type. If not, something is inconsistent for that
# one game and needs a manual look before we trust it.
game_type_check = df_pbp.groupby(['match_id', 'set_id', 'game_id'])['game_type'].nunique()
mixed_games = game_type_check[game_type_check > 1]
print('games with inconsistent game_type:', len(mixed_games))
print(mixed_games)

game_type
normal_game    1215567
tie_break        39173
Name: count, dtype: int64
games with inconsistent game_type: 1
match_id  set_id  game_id
12041957  3       1          2
Name: game_type, dtype: int64


In [42]:
# اصلاح: نوع گیم باید یکبار برای کل گیم تعیین بشه، نه پوینت‌به‌پوینت —
# چون اولین پوینت یه تای‌بریک (0-0) با اولین پوینت گیم عادی یکسان به‌نظر میاد.
# قانون درست: اگه حتی یک پوینت از این گیم شکل تای‌بریکی داشت، کل گیم تای‌بریکه.
game_type_per_game = df_pbp.groupby(['match_id', 'set_id', 'game_id'])['game_type'].agg(
    lambda s: 'tie_break' if (s == 'tie_break').any() else 'normal_game'
)

df_pbp = df_pbp.drop(columns='game_type').merge(
    game_type_per_game.rename('game_type'),
    on=['match_id', 'set_id', 'game_id']
)

# تأیید: دیگه نباید هیچ گیم مخلوطی بمونه
check_again = df_pbp.groupby(['match_id', 'set_id', 'game_id'])['game_type'].nunique()
print('گیم‌های همچنان مخلوط:', (check_again > 1).sum())   # باید 0 بشه

گیم‌های همچنان مخلوط: 0


In [43]:
# ---------------------------------------------------------------------------
# 4) Validate tie-break games: scores must never decrease point-to-point,
#    and the game must end at >=7 points with at least a 2-point margin.
# ---------------------------------------------------------------------------
tie_break_data = df_pbp[df_pbp['game_type'] == 'tie_break'].copy()
tie_break_data['home_point'] = pd.to_numeric(tie_break_data['home_point'])
tie_break_data['away_point'] = pd.to_numeric(tie_break_data['away_point'])

def validate_tie_break(group):
    group = group.sort_values('point_id')
    if (group['home_point'].diff() < 0).any() or (group['away_point'].diff() < 0).any():
        return 'invalid_sequence'
    last = group.iloc[-1]
    winner_score = max(last['home_point'], last['away_point'])
    score_diff = abs(last['home_point'] - last['away_point'])
    if winner_score < 7:
        return 'invalid_final_score'
    if score_diff < 2:
        return 'invalid_difference'
    return 'valid'

def fix_score_sequence(group):
    # A score that decreases from the previous point is almost certainly
    # a scraping glitch (e.g. a stale/duplicate point slipped in) rather
    # than a real rule violation, so we carry the previous max forward.
    group = group.sort_values('point_id').reset_index(drop=True)
    for i in range(1, len(group)):
        if group.loc[i, 'home_point'] < group.loc[i - 1, 'home_point']:
            group.loc[i, 'home_point'] = group.loc[i - 1, 'home_point']
        if group.loc[i, 'away_point'] < group.loc[i - 1, 'away_point']:
            group.loc[i, 'away_point'] = group.loc[i - 1, 'away_point']
    return group

valid_tb_parts, invalid_tb_parts = [], []
for key, group in tie_break_data.groupby(['match_id', 'set_id', 'game_id']):
    result = validate_tie_break(group)
    if result == 'invalid_sequence':
        group = fix_score_sequence(group)
        result = validate_tie_break(group)
    if result == 'valid':
        valid_tb_parts.append(group)
    else:
        invalid_tb_parts.append(group)

tie_break_valid = pd.concat(valid_tb_parts, ignore_index=True) if valid_tb_parts else pd.DataFrame()
tie_break_invalid = pd.concat(invalid_tb_parts, ignore_index=True) if invalid_tb_parts else pd.DataFrame()
print('tie-break valid rows:', tie_break_valid.shape[0], '| invalid rows:', tie_break_invalid.shape[0])

tie-break valid rows: 39136 | invalid rows: 38


In [44]:
# ---------------------------------------------------------------------------
# 5) Validate normal games: points must follow the real 0/15/30/40 ->
#    game progression, with correct deuce/advantage handling, exactly one
#    side scoring per point, and a legal final score.
# ---------------------------------------------------------------------------
normal_game_data = df_pbp[df_pbp['game_type'] == 'normal_game'].copy()

SCORE_ORDER = {'0': 0, '15': 1, '30': 2, '40': 3}
VALID_WINNING_SCORES = {('40', '0'), ('40', '15'), ('40', '30'),
                         ('0', '40'), ('15', '40'), ('30', '40'),
                         ('A', '40'), ('40', 'A')}

def validate_normal_game(group):
    group = group.sort_values('point_id')
    scores = list(zip(group['home_point'], group['away_point']))

    for i in range(1, len(scores)):
        prev, curr = scores[i - 1], scores[i]

        if prev in (('A', '40'), ('40', 'A')):
            if curr == ('40', '40'):
                continue
            return 'invalid_sequence'

        if prev == ('40', '40'):
            if curr in (('A', '40'), ('40', 'A'), ('40', '40')):
                continue
            return 'invalid_sequence'

        if curr[0] == 'A' or curr[1] == 'A':
            return 'invalid_sequence'

        home_change = SCORE_ORDER[curr[0]] - SCORE_ORDER[prev[0]]
        away_change = SCORE_ORDER[curr[1]] - SCORE_ORDER[prev[1]]

        if home_change < 0 or away_change < 0:
            return 'invalid_sequence'
        if home_change > 0 and away_change > 0:
            return 'invalid_sequence'
        if home_change == 0 and away_change == 0:
            return 'invalid_sequence'

    if scores[-1] in VALID_WINNING_SCORES:
        return 'valid'
    return 'invalid_final_score'

normal_validation = normal_game_data.groupby(
    ['match_id', 'set_id', 'game_id']
).apply(validate_normal_game, include_groups=False)
print(normal_validation.value_counts())

valid_keys = normal_validation[normal_validation == 'valid'].index
invalid_keys = normal_validation[normal_validation != 'valid'].index

normal_valid = (normal_game_data.set_index(['match_id', 'set_id', 'game_id'])
                .loc[valid_keys].reset_index())
normal_invalid = (normal_game_data.set_index(['match_id', 'set_id', 'game_id'])
                   .loc[invalid_keys].reset_index())
print('normal valid rows:', normal_valid.shape[0], '| invalid rows:', normal_invalid.shape[0])

valid                  224709
invalid_final_score       264
invalid_sequence           82
Name: count, dtype: int64
normal valid rows: 1213280 | invalid rows: 2286


In [45]:
# ---------------------------------------------------------------------------
# 6) Combine into final valid / invalid tables
# ---------------------------------------------------------------------------
final_pbp = pd.concat([normal_valid, tie_break_valid], ignore_index=True)
invalid_pbp = pd.concat([normal_invalid, tie_break_invalid], ignore_index=True)

print('FINAL valid rows:', final_pbp.shape[0])
print('FINAL invalid rows:', invalid_pbp.shape[0])
print('invalid share:', round(100 * invalid_pbp.shape[0] / df_pbp.shape[0], 3), '%')

flat_tables['point_by_point'] = final_pbp
flat_tables['point_by_point_invalid'] = invalid_pbp


FINAL valid rows: 1252416
FINAL invalid rows: 2324
invalid share: 0.185 %


#Cleaning the Power table

In [46]:
# ---------------------------------------------------------------------------
# 1) Read Tennis Power data
# ---------------------------------------------------------------------------

df_power = flat_tables['power'].copy()

print("Power shape:", df_power.shape)
print("Power columns:")
print(df_power.columns.tolist())

df_power.head(10)

Power shape: (469677, 6)
Power columns:
['match_id', 'set_num', 'game_num', 'value', 'break_occurred', 'source_day']


,match_id,set_num,game_num,value,break_occurred,source_day
0,11998445,1,1,-52.80,False,20240201
1,11998445,1,2,48.14,False,20240201
2,11998445,1,3,-51.62,False,20240201
3,11998445,1,4,10.00,False,20240201
4,11998445,1,5,26.60,True,20240201
5,11998445,1,6,10.00,False,20240201
6,11998445,1,7,-10.00,False,20240201
7,11998445,1,8,-59.30,True,20240201
8,11998445,1,9,-33.02,False,20240201
9,11998445,1,10,36.82,False,20240201


In [47]:
dup_count = df_power.duplicated(subset=['match_id', 'set_num', 'game_num'], keep=False).sum()
print('ردیف‌های درگیر در دوبله:', dup_count)

ردیف‌های درگیر در دوبله: 452877


In [48]:
# یه match_id با چند source_day پیدا کن و ببین آیا مقدار 'value' واقعاً یکسانه
multi_day_power = df_power.groupby('match_id')['source_day'].nunique()
sample_match = multi_day_power[multi_day_power > 1].index[0]
print('نمونه match_id:', sample_match)

sub = df_power[df_power['match_id'] == sample_match].sort_values(['set_num', 'game_num', 'source_day'])
print(sub[['set_num', 'game_num', 'value', 'break_occurred', 'source_day']].head(20))

نمونه match_id: 11998445
      set_num  game_num  value  break_occurred source_day
0           1         1 -52.80           False   20240201
5894        1         1 -52.80           False   20240202
1           1         2  48.14           False   20240201
5895        1         2  48.14           False   20240202
2           1         3 -51.62           False   20240201
5896        1         3 -51.62           False   20240202
3           1         4  10.00           False   20240201
5897        1         4  10.00           False   20240202
4           1         5  26.60            True   20240201
5898        1         5  26.60            True   20240202
5           1         6  10.00           False   20240201
5899        1         6  10.00           False   20240202
6           1         7 -10.00           False   20240201
5900        1         7 -10.00           False   20240202
7           1         8 -59.30            True   20240201
5901        1         8 -59.30            True 

In [49]:
# برای هر (match_id, set_num, game_num) که بیش از یک source_day داره،
# ببین آیا value واقعاً همیشه ثابت مونده یا نه
dup_rows = df_power[df_power.duplicated(subset=['match_id', 'set_num', 'game_num'], keep=False)]
value_consistency = dup_rows.groupby(['match_id', 'set_num', 'game_num'])['value'].nunique()
print('تعداد گروه‌هایی که value بینشون فرق داره:', (value_consistency > 1).sum(), 'از', len(value_consistency))

تعداد گروه‌هایی که value بینشون فرق داره: 9062 از 213781


In [50]:
# نمونه‌ای از گروه‌هایی که واقعاً value متفاوت دارن
changed_keys = value_consistency[value_consistency > 1].index[:5]

for key in changed_keys:
    mid, sn, gn = key
    print(df_power[(df_power['match_id']==mid) & (df_power['set_num']==sn) & (df_power['game_num']==gn)]
          [['match_id','set_num','game_num','value','break_occurred','source_day']])
    print('---')

        match_id  set_num  game_num  value  break_occurred source_day
117979  12075951        2         1   34.2            True   20240217
122391  12075951        2         1   10.0            True   20240218
---
        match_id  set_num  game_num  value  break_occurred source_day
117980  12075951        2         2  30.54           False   20240217
122392  12075951        2         2  25.48           False   20240218
---
        match_id  set_num  game_num  value  break_occurred source_day
206665  12087266        1         6   10.0           False   20240229
216473  12087266        1         6    0.0            True   20240301
---
        match_id  set_num  game_num  value  break_occurred source_day
206667  12087266        1         8   10.0           False   20240229
216475  12087266        1         8   18.0           False   20240301
---
        match_id  set_num  game_num  value  break_occurred source_day
206670  12087266        2         2  -10.0           False   20240229
2164

In [51]:
df_power_sorted = df_power.sort_values('source_day')
before = df_power_sorted.shape[0]
df_power_clean = df_power_sorted.drop_duplicates(subset=['match_id', 'set_num', 'game_num'], keep='last')
after = df_power_clean.shape[0]
print(before, '->', after)

flat_tables['power'] = df_power_clean

469677 -> 230581


In [52]:
df_power = flat_tables['power']
print(df_power['value'].describe())

count    230581.000000
mean         -0.475850
std          37.226038
min        -100.000000
25%         -27.200000
50%           6.000000
75%          24.820000
max         186.000000
Name: value, dtype: float64


In [53]:
print(df_power['break_occurred'].value_counts(normalize=True))

break_occurred
False    0.665636
True     0.334364
Name: proportion, dtype: float64


In [54]:
# برای هر گیم عادی در point_by_point، آیا سرویس‌زننده همون کسی بود که امتیاز نهایی گیم رو برد؟
pbp = flat_tables['point_by_point']
normal_games = pbp[pbp['game_type'] == 'normal_game']

last_points = normal_games.sort_values('point_id').groupby(['match_id', 'set_id', 'game_id']).last()
last_points['real_break'] = last_points['serving'] != last_points['scoring']

real_break_rate = last_points['real_break'].mean()
print('نرخ واقعی بریک سرویس (از point_by_point):', round(real_break_rate * 100, 2), '%')

نرخ واقعی بریک سرویس (از point_by_point): 32.69 %


In [55]:
# آیا جهت value با برنده‌ی واقعی گیم (از point_by_point) هماهنگه؟
last_points_reset = last_points.reset_index()
last_points_reset['home_won_game'] = last_points_reset['scoring'] == 1

merged_power = df_power.merge(
    last_points_reset[['match_id', 'set_id', 'game_id', 'home_won_game']],
    left_on=['match_id', 'set_num', 'game_num'],
    right_on=['match_id', 'set_id', 'game_id'],
    how='inner'
)

# وقتی home گیم رو برده، انتظار داریم value معمولا مثبت باشه (یا برعکس - بستگی به قرارداد sofascore)
print(merged_power.groupby('home_won_game')['value'].mean())

home_won_game
False   -31.213370
True     29.579674
Name: value, dtype: float64


# cleaning the odds table

In [56]:
# ---------------------------------------------------------------------------
# 1) Read Odds data
# ---------------------------------------------------------------------------

df_odds = flat_tables['odds'].copy()

print("Odds shape:", df_odds.shape)

print("Odds columns:")
print(df_odds.columns.tolist())

df_odds.head(10)

Odds shape: (60946, 12)
Odds columns:
['match_id', 'market_id', 'market_name', 'is_live', 'suspended', 'initial_fractional_value', 'fractional_value', 'choice_name', 'choice_source_id', 'winnig', 'change', 'source_day']


,match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change,source_day
0,11974053,1,full_time,False,False,1/4,1/5,1,1994690951,True,-1,20240201
1,11974053,1,full_time,False,False,11/4,10/3,2,1994691017,False,1,20240201
2,11974066,1,full_time,False,False,14/1,14/1,1,1990515041,False,0,20240201
3,11974066,1,full_time,False,False,1/33,3/100,2,1990515081,True,0,20240201
4,11998445,1,full_time,False,False,1/1,4/5,1,1985921173,False,-1,20240201
5,11998445,1,full_time,False,False,4/5,1/1,2,1985921202,True,1,20240201
6,11998445,11,first_set_winner,False,False,10/11,4/5,1,1985924574,False,-1,20240201
7,11998445,11,first_set_winner,False,False,10/11,1/1,2,1985924691,True,1,20240201
8,11998445,12,total_games_won,False,False,5/6,5/6,Over,1987198066,True,0,20240201
9,11998445,12,total_games_won,False,False,5/6,5/6,Under,1987198051,False,0,20240201


In [57]:
dup_count = df_odds.duplicated(subset=['match_id', 'market_id', 'choice_name'], keep=False).sum()
print('ردیف‌های درگیر در دوبله:', dup_count)

ردیف‌های درگیر در دوبله: 59970


#Question 1: How many tennis players are included in the dataset?

In [58]:
# Stack home and away player info into one column set
home = master[['home_team_player_id', 'home_team_height']]
away = master[['away_team_player_id', 'away_team_height']]

home.columns = ['player_id', 'height']
away.columns = ['player_id', 'height']

players = pd.concat([home, away])

# Remove duplicate players (each player appears once per match they played)
players = players.drop_duplicates(subset='player_id')

print('Number of unique players:', players['player_id'].nunique())

Number of unique players: 2644


#Question 2: What's the average height of the players?

In [59]:
print('Average height (meters):', players['height'].mean())
print('Average height (cm):', players['height'].mean() * 100)
print('Players with known height:', players['height'].notna().sum(), 'out of', players.shape[0])

Average height (meters): 1.8213743356112377
Average height (cm): 182.13743356112377
Players with known height: 1317 out of 2645


#Question 3: Which player has the highest number of wins?

In [60]:
# winner's player_id for each match
home_win = master.loc[master['winner_code'] == 1, ['home_team_player_id', 'home_team_full_name']]
away_win = master.loc[master['winner_code'] == 2, ['away_team_player_id', 'away_team_full_name']]

home_win.columns = ['player_id', 'name']
away_win.columns = ['player_id', 'name']

winners = pd.concat([home_win, away_win])

win_counts = winners['player_id'].value_counts()
print(win_counts.head(10))   # top 10 players by wins

top_id = win_counts.index[0]
top_name = winners.loc[winners['player_id'] == top_id, 'name'].iloc[0]
print('Most wins:', top_name, '-', win_counts.iloc[0], 'wins')

player_id
50901.0     29
231620.0    22
202572.0    21
230049.0    21
82133.0     20
238867.0    19
58369.0     19
205282.0    19
338890.0    18
186885.0    18
Name: count, dtype: int64
Most wins: Popko, Dmitry - 29 wins


#Question 4: What is the longest match recorded in terms of duration?

In [61]:
duration_cols = ['time_period_1', 'time_period_2', 'time_period_3',
                  'time_period_4', 'time_period_5']

master['total_duration_sec'] = master[duration_cols].sum(axis=1, skipna=True)

top5 = master.sort_values('total_duration_sec', ascending=False).head(5)
print(top5[['match_id', 'total_duration_sec', 'home_team_full_name', 'away_team_full_name']])

       match_id total_duration_sec   home_team_full_name away_team_full_name
2942   12063611           336790.0      Shapatava, Sofia   Branstine, Carson
2941   12063587           320230.0  Podlinska, Marcelina       Vargová, Nina
14514  12185562           177131.0         Gavrilă, Oana     Alves, Carolina
8131   12121829           173415.0        Carou, Ignacio       Rybakov, Alex
1923   12054403           163883.0      Gjorcheska, Lina          Würth Tara


In [62]:
# ببینیم توزیع کلی time_period_1 چه شکلیه (نه فقط بیشینه‌ش)
print(master['time_period_1'].describe())

count     10327.000000
mean       2867.427520
std        4995.894419
min           2.000000
25%        2042.000000
50%        2516.000000
75%        3147.500000
max      172605.000000
Name: time_period_1, dtype: float64


In [63]:
# یه بازی با مدت "معقول" (مثلاً بین 20 دقیقه تا 2 ساعت، یعنی 1200 تا 7200 ثانیه) پیدا کن
# و ببینیم بقیه‌ی ستون‌هاش با فرض "duration" جور درمیاد یا نه
normal_case = master[(master['time_period_1'] > 1200) & (master['time_period_1'] < 7200)].iloc[0]
print(normal_case[['match_id', 'time_period_1', 'time_period_2', 'time_period_3',
                    'home_team_score_period_1', 'away_team_score_period_1']])

match_id                    11998445
time_period_1                 3259.0
time_period_2                 2639.0
time_period_3                 4202.0
home_team_score_period_1         5.0
away_team_score_period_1         7.0
Name: 2, dtype: object


In [64]:
THRESHOLD = 14400  # 4 hours per set, generous upper bound

duration_cols = ['time_period_1', 'time_period_2', 'time_period_3',
                  'time_period_4', 'time_period_5']

# مقادیر بالاتر از آستانه رو ببینیم چقدرن، قبل از حذف
for c in duration_cols:
    n_bad = (master[c] > THRESHOLD).sum()
    print(c, ': تعداد مقدار مشکوک =', n_bad)

time_period_1 : تعداد مقدار مشکوک = 25
time_period_2 : تعداد مقدار مشکوک = 52
time_period_3 : تعداد مقدار مشکوک = 25
time_period_4 : تعداد مقدار مشکوک = 0
time_period_5 : تعداد مقدار مشکوک = 0


In [65]:
# نسخه‌ی تمیزشده: مقادیر مشکوک رو NaN می‌کنیم، بعد جمع می‌زنیم
master_time_clean = master[duration_cols].where(master[duration_cols] <= THRESHOLD)
master['total_duration_sec'] = master_time_clean.sum(axis=1, skipna=True)

# فقط بازی‌هایی که حداقل یه ست معتبر دارن رو در نظر بگیر
valid_duration = master[master['total_duration_sec'] > 0]

top5 = valid_duration.sort_values('total_duration_sec', ascending=False).head(5)
print(top5[['match_id', 'total_duration_sec', 'home_team_full_name', 'away_team_full_name']])

       match_id total_duration_sec   home_team_full_name  \
36     11999209            26968.0         Alves, Mateus   
2333   12058493            24326.0           Lukas, Tena   
11166  12151527            23266.0   Paganetti, Vittoria   
13426  12170619            20523.0         Piatti, Rocco   
15348  12191915            20385.0  Cid Subervi, Roberto   

         away_team_full_name  
36     Ugo Carabelli, Camilo  
2333          Hercog, Polona  
11166     Radwańska, Urszula  
13426           Gola, Andrea  
15348         Kozlov, Stefan  


In [66]:
# columns holding the duration (in seconds) of each set
duration_cols = ['time_period_1', 'time_period_2', 'time_period_3',
                  'time_period_4', 'time_period_5']

# a small number of rows have impossible values (e.g. 48 hours for one set),
# so we ignore anything above 4 hours (14400 seconds) per set as bad data
THRESHOLD = 14400

clean_durations = master[duration_cols].where(master[duration_cols] <= THRESHOLD)
master['total_duration_sec'] = clean_durations.sum(axis=1, skipna=True)

# only look at matches that ended up with a valid duration
valid = master[master['total_duration_sec'] > 0]

# the single longest match
longest = valid.sort_values('total_duration_sec', ascending=False).iloc[0]

print('Longest match:', longest['home_team_full_name'], 'vs', longest['away_team_full_name'])
print('Duration (seconds):', longest['total_duration_sec'])
print('Duration (hours):', round(longest['total_duration_sec'] / 3600, 2))

Longest match: Alves, Mateus vs Ugo Carabelli, Camilo
Duration (seconds): 26968.0
Duration (hours): 7.49


#Question 5: How many sets are typically played in a tennis match?

In [67]:
# only keep matches that actually finished with full set-by-set data:
# - has a winner
# - not flagged as retirement/incomplete
# - not a "final result only" record (no set details)
completed = master[
    (master['winner_code'].notna()) &
    (master['likely_retirement_or_incomplete'] == False) &
    (master['final_result_only'] == False)
]

# count how many sets have a score recorded
score_cols = ['home_team_score_period_1', 'home_team_score_period_2',
              'home_team_score_period_3', 'home_team_score_period_4',
              'home_team_score_period_5']

num_sets = completed[score_cols].notna().sum(axis=1)

print(num_sets.value_counts().sort_index())
print('Most common number of sets:', num_sets.mode()[0])
print('Average number of sets:', num_sets.mean())

0      80
1      83
2    9501
3    3689
Name: count, dtype: int64
Most common number of sets: 2
Average number of sets: 2.2580693477121248


In [68]:
weird = completed[num_sets <= 1]
print(weird[['match_id', 'winner_code', 'final_result_only',
             'home_team_score_period_1', 'away_team_score_period_1',
             'home_team_score_period_2', 'away_team_score_period_2']].head(10))

     match_id  winner_code  final_result_only  home_team_score_period_1  \
0    11974053          1.0              False                       NaN   
1    11974066          2.0              False                       NaN   
44   12001976          2.0              False                       NaN   
63   12002078          1.0              False                       NaN   
64   12002109          2.0              False                       NaN   
65   12002112          2.0              False                       NaN   
221  12022528          2.0              False                       NaN   
243  12023465          1.0              False                       6.0   
285  11974052          2.0              False                       NaN   
286  11974067          2.0              False                       NaN   

     away_team_score_period_1  home_team_score_period_2  \
0                         NaN                       NaN   
1                         NaN                       NaN 

In [69]:
# اگه فرضیه‌ی Walkover تأیید شد، این‌ها رو هم فیلتر کن
final_completed = completed[num_sets >= 2]
final_num_sets = final_completed[score_cols].notna().sum(axis=1)

print(final_num_sets.value_counts().sort_index())
print('Most common:', final_num_sets.mode()[0])
print('Average:', round(final_num_sets.mean(), 2))

2    9501
3    3689
Name: count, dtype: int64
Most common: 2
Average: 2.28


# Question 6: Which country has produced the most successful tennis players?

In [75]:
# (defined here as: total match wins grouped by player's country)

home_win = master.loc[master['winner_code'] == 1, ['home_team_player_id', 'home_team_country']]
away_win = master.loc[master['winner_code'] == 2, ['away_team_player_id', 'away_team_country']]

home_win.columns = ['player_id', 'country']
away_win.columns = ['player_id', 'country']

winners = pd.concat([home_win, away_win])

wins_by_country = winners['country'].value_counts()
print(wins_by_country.head(10))
print('Country with most wins:', wins_by_country.index[0], '-', wins_by_country.iloc[0], 'wins')

country
France            990
Italy             935
USA               910
Russia            626
Argentina         549
Germany           529
Japan             523
Spain             480
Australia         437
United Kingdom    360
Name: count, dtype: int64
Country with most wins: France - 990 wins


In [76]:
# مجموع بازی (نه فقط برد) به ازای هر کشور
home_all = master[['home_team_country']].rename(columns={'home_team_country': 'country'})
away_all = master[['away_team_country']].rename(columns={'away_team_country': 'country'})
all_matches_by_country = pd.concat([home_all, away_all])['country'].value_counts()

win_rate = (wins_by_country / all_matches_by_country * 100).dropna().sort_values(ascending=False)

# آستانه‌ی بالاتر: حداقل 300 بازی، برای مقایسه‌ی معنادار آماری
reliable = win_rate[all_matches_by_country >= 300].sort_values(ascending=False)
print(reliable.head(10))

country
Argentina         52.285714
Czech Republic    51.724138
Australia         50.932401
USA               50.611791
Poland            50.299401
Slovakia          50.253807
Russia            50.000000
Romania           50.000000
China             49.400000
Belgium           49.240122
Name: count, dtype: float64


 Q16: Which player has the highest winning percentage against top-10 ranked opponents?

In [77]:
home = master[['home_team_player_id', 'home_team_full_name', 'away_team_current_rank']].copy()
home.columns = ['player_id', 'name', 'opponent_rank']
home['won'] = master['winner_code'] == 1

away = master[['away_team_player_id', 'away_team_full_name', 'home_team_current_rank']].copy()
away.columns = ['player_id', 'name', 'opponent_rank']
away['won'] = master['winner_code'] == 2

vs_all = pd.concat([home, away])

vs_top10 = vs_all[vs_all['opponent_rank'] <= 10]

summary = vs_top10.groupby('player_id').agg(
    name=('name', 'first'),
    matches=('won', 'size'),
    wins=('won', 'sum')
)
summary['win_pct'] = summary['wins'] / summary['matches'] * 100

result = summary.sort_values('win_pct', ascending=False)
print(result.head(20))

                           name  matches  wins     win_pct
player_id                                                 
19578.0       Kerber, Angelique        1     1  100.000000
127208.0       Altmaier, Daniel        1     1  100.000000
69208.0        Paolini, Jasmine        1     1  100.000000
87690.0        Thompson, Jordan        1     1  100.000000
90664.0      Kalinina, Anhelina        1     1  100.000000
64580.0            Ćorić, Borna        1     1  100.000000
50641.0            Vekić, Donna        1     1  100.000000
59702.0    Haddad Maia, Beatriz        1     1  100.000000
47603.0        Monteiro, Thiago        1     1  100.000000
36067.0         Tsurenko, Lesia        1     1  100.000000
228272.0           Swiatek, Iga        3     3  100.000000
256104.0        Volynets, Katie        1     1  100.000000
262594.0       Avanesyan, Elina        1     1  100.000000
289233.0            Nardi, Luca        1     1  100.000000
406728.0        Michelsen, Alex        1     1  100.0000

In [82]:
# Custom 19: Does winning the first set predict winning the match?

master['home_won_set1'] = master['home_team_score_period_1'] > master['away_team_score_period_1']
master['home_won_match'] = master['winner_code'] == 1

# only matches with a known first-set score and a known winner
valid = master.dropna(subset=['home_team_score_period_1', 'away_team_score_period_1', 'winner_code'])

match_rate = (valid['home_won_set1'] == valid['home_won_match']).mean()
print('Set 1 winner also wins the match:', round(match_rate * 100, 1), '%')

Set 1 winner also wins the match: 84.8 %


In [83]:
# Custom 20: Do seeded players actually perform better?

# only matches with a known winner
completed = master[master['winner_code'].notna()]

home = completed[['home_team_seed']].copy()
home['seeded'] = home['home_team_seed'].notna()
home['won'] = completed['winner_code'] == 1

away = completed[['away_team_seed']].copy()
away['seeded'] = away['away_team_seed'].notna()
away['won'] = completed['winner_code'] == 2

seed_data = pd.concat([home[['seeded', 'won']], away[['seeded', 'won']]])

print(seed_data.groupby('seeded')['won'].mean())

seeded
False    0.480257
True     0.553259
Name: won, dtype: float64
